# Instalando a bibloteca nba_api

In [ ]:
pip install --upgrade nba-api

# Importando biblotecas e funções de nba_api


In [2]:
from nba_api.stats.endpoints import leaguedashteamstats
from nba_api.stats.endpoints import commonplayoffseries
from nba_api.stats.endpoints import leaguedashteamshotlocations
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguedashplayerstats
import pandas as pd
import numpy as np


# Filtrando os times campeões

In [3]:
seasons = pd.Series(["2015-16","2016-17","2017-18","2018-19","2019-20","2020-21",
           "2021-22","2022-23","2023-24","2024-25"])

teams_Series = ["CLEVELAND CAVALIERS", "GOLDEN STATE WARRIORS",
                      "GOLDEN STATE WARRIORS", "TORONTO RAPTORS","LOS ANGELES LAKERS",
                      "MILWAUKEE BUCKS", "GOLDEN STATE WARRIORS", "BOSTON CELTICS",
                        "DENVER NUGGETS", "OKLAHOMA CITY THUNDER"]

df_last_10_champions = pd.DataFrame(index=seasons)
df_last_10_champions["TEAMS"] = teams_Series



# Filtrando todos os times por id e por nome


In [4]:
nba_teams = [
    # Conferência Leste
    "ATLANTA HAWKS", "BOSTON CELTICS", "BROOKLYN NETS", "CHARLOTTE HORNETS",
    "CHICAGO BULLS", "CLEVELAND CAVALIERS", "DETROIT PISTONS", "INDIANA PACERS",
    "MIAMI HEAT", "MILWAUKEE BUCKS", "NEW YORK KNICKS", "ORLANDO MAGIC",
    "PHILADELPHIA 76ERS", "TORONTO RAPTORS", "WASHINGTON WIZARDS",

    # Conferência Oeste
    "DALLAS MAVERICKS", "DENVER NUGGETS", "GOLDEN STATE WARRIORS", "HOUSTON ROCKETS",
    "LOS ANGELES CLIPPERS", "LOS ANGELES LAKERS", "MEMPHIS GRIZZLIES",
    "MINNESOTA TIMBERWOLVES", "NEW ORLEANS PELICANS", "OKLAHOMA CITY THUNDER",
    "PHOENIX SUNS", "PORTLAND TRAIL BLAZERS", "SACRAMENTO KINGS",
    "SAN ANTONIO SPURS", "UTAH JAZZ"
]


def find_ID_by_name(team_name):
    team_infos = teams.find_teams_by_full_name(team_name)
    return team_infos[0]["id"]

def find_abb_by_name(team_name):
    team_infos = teams.find_teams_by_full_name(team_name)
    return team_infos[0]["abbreviation"]

def getting_ID_row_by_name(teams_Series, df_teams):
    dict_times_ids = {}
    for time in teams_Series:
        dict_times_ids[time] = find_ID_by_name(time)

    for time in df_teams["TEAMS"]:
        for k in dict_times_ids.keys():
            if time == k:
                df_teams.loc[df_teams["TEAMS"] == time, "TEAM_ID"] = str(dict_times_ids[time])

# Busca de parâmetros

In [ ]:

#Ofensive stats
def get_ofensive_stats(season,season_type):
  advanced = leaguedashteamstats.LeagueDashTeamStats(
      season = season,
      measure_type_detailed_defense = 'Advanced',
      per_mode_detailed="PerGame",
      season_type_all_star = season_type).get_data_frames()[0]
  colums_advanced =  ['TEAM_ID', 'TEAM_NAME','OFF_RATING', 'NET_RATING', 'AST_PCT','AST_TO', 'AST_RATIO', 'TM_TOV_PCT', 'EFG_PCT', 'TS_PCT','PACE','E_PACE']

  four_factores = leaguedashteamstats.LeagueDashTeamStats(
      season = season,
      measure_type_detailed_defense = 'Four Factors',
      per_mode_detailed="PerGame",
      season_type_all_star = season_type).get_data_frames()[0]
  colums_fourfactores = ["TEAM_ID","FTA_RATE", "OREB_PCT"]

  df_offensive = pd.merge(advanced[colums_advanced], four_factores[colums_fourfactores], on = 'TEAM_ID')

  return df_offensive

#Defensive stats
def get_defensive_stats(season,season_type):
  Defense = leaguedashteamstats.LeagueDashTeamStats(
      season = season,
      measure_type_detailed_defense = 'Defense',
      per_mode_detailed="PerGame",
      season_type_all_star = season_type).get_data_frames()[0]

  colunms_defense = ['TEAM_ID', 'TEAM_NAME','DEF_RATING','STL', 'BLK', 'DREB_PCT','OPP_PTS_2ND_CHANCE','OPP_PTS_PAINT',]

  Opponent = leaguedashteamstats.LeagueDashTeamStats(
      season = season,
      measure_type_detailed_defense = 'Opponent',
      per_mode_detailed="PerGame",
      season_type_all_star = season_type).get_data_frames()[0]

  colums_opponent =['TEAM_ID','OPP_FGM',
       'OPP_FGA', 'OPP_FG_PCT', 'OPP_FG3M', 'OPP_FG3A', 'OPP_FG3_PCT','OPP_FTA','OPP_REB',
       'OPP_AST', 'OPP_TOV','OPP_PTS']

  df_defensive = pd.merge(Defense[colunms_defense], Opponent[colums_opponent], on = 'TEAM_ID')

  return df_defensive

#Bench points%
def get_bench_points(season,season_type):
  bench = leaguedashteamstats.LeagueDashTeamStats(
      season = season,
      measure_type_detailed_defense='Base',
      starter_bench_nullable = 'Bench',
      per_mode_detailed="PerGame",
      season_type_all_star = season_type).get_data_frames()[0]

  team = leaguedashteamstats.LeagueDashTeamStats(
      season = season,
      per_mode_detailed="PerGame",
      season_type_all_star = season_type,
      measure_type_detailed_defense='Base').get_data_frames()[0]


  cols_bench = ['TEAM_ID','TEAM_NAME','PTS']
  cols_team = ['TEAM_ID','PTS']
  bench = bench[cols_bench]
  bench = bench.rename(columns={'PTS': 'BENCH_PTS_PER_GAME'})
  team = team[cols_team]
  df_total = pd.merge(bench, team, on='TEAM_ID')
  df_total['BENCH_PTS_PCT'] = df_total['BENCH_PTS_PER_GAME'] / df_total['PTS']
  dftotal = df_total.drop(columns = {'TEAM_ID','BENCH_PTS_PCT'})

  return df_total


#Shot profile/ Team identity
def get_shot_locations(season,season_type):
  scoring_stats = leaguedashteamstats.LeagueDashTeamStats(
      season = season,
      measure_type_detailed_defense = 'Scoring',
      per_mode_detailed="PerGame",
      season_type_all_star = season_type).get_data_frames()[0]
  colunas_selecionadas = ['TEAM_ID','PCT_PTS_2PT' , 'PCT_FGA_2PT', 'PCT_PTS_3PT','PCT_FGA_3PT']

  shot_locations = leaguedashteamshotlocations.LeagueDashTeamShotLocations(
        season=season,
        measure_type_simple='Base',
        per_mode_detailed="PerGame",
        season_type_all_star = season_type).get_data_frames()[0]

  # FG% 3pts
  fgm_3pt = shot_locations['Left Corner 3' , 'FGM'] + shot_locations['Right Corner 3' , 'FGM'] + shot_locations['Above the Break 3' , 'FGM']
  fga_3pt = shot_locations['Left Corner 3' , 'FGA'] + shot_locations['Right Corner 3' , 'FGA'] + shot_locations['Above the Break 3' , 'FGA']
  fgpct_3 = fgm_3pt / fga_3pt

  df_total = pd.DataFrame()
  df_total["Team_ID"] = shot_locations[( "","TEAM_ID")]
  df_total["Team_NAME"] = shot_locations[( "","TEAM_NAME")]
  df_total["Rim_PCT"] = shot_locations["Restricted Area", "FG_PCT"]
  df_total["RIm_FGA"] = shot_locations["Restricted Area", "FGA"]
  df_total["Paint_PCT"] = shot_locations["In The Paint (Non-RA)", "FG_PCT"]
  df_total["Paint_FGA"] = shot_locations["In The Paint (Non-RA)", "FGA"]
  df_total["Mid-Range_PCT"] = shot_locations["Mid-Range", "FG_PCT"]
  df_total["Mid-Range_FGA"] = shot_locations["Mid-Range", "FGA"]
  df_total["PCT_PTS_2PT"] = scoring_stats["PCT_PTS_2PT"]
  df_total["3 PT_PCT"] = fgpct_3
  df_total["3 PT_FGA"] = fga_3pt
  df_total["PCT_PTS_3PT"] = scoring_stats["PCT_PTS_3PT"]

  return df_total

def get_usage_stars(season, season_type,min_games=30):

    # Criando um df_teams (auxiliar)
    df_teams = pd.DataFrame()
    df_teams["TEAMS"] = nba_teams
    getting_ID_row_by_name(df_teams["TEAMS"], df_teams)
    df_teams

    df = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star=season_type,
        per_mode_detailed='PerGame',
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]

    cols = ["PLAYER_NAME", "TEAM_ID", "TEAM_ABBREVIATION", "GP", "USG_PCT"]
    metric = "USG_PCT"

    df = df[cols]
    df_filtered = df[df['GP'] >= min_games].copy()
    team_ids = df_teams["TEAM_ID"].astype(int)

    # Criando listas que servirão de colunas para o df final
    leader1_name = []
    leader2_name = []
    leader1_usg = []
    leader2_usg = []


    for team_id in team_ids:
        team_df = df_filtered[df_filtered['TEAM_ID'] == team_id]

        # Cria um DataFrame ordenado pelos 2 jogadores com mais USG%
        top_2 = team_df.sort_values(by=metric, ascending=False).head(2)

        players_name = []
        players_usg = []
        for index, player in top_2.iterrows(): # Index iterando as linhas
            usg_val = player[metric]
            players_name.append(player['PLAYER_NAME'])
            players_usg.append(round(usg_val, 1))

        leader1_usg.append(players_usg[0])
        leader2_usg.append(players_usg[1])
        leader1_name.append(players_name[0])
        leader2_name.append(players_name[1])

    df_results = pd.DataFrame()
    df_results["TEAM_NAME"] = nba_teams
    df_results["LEADER 1"], df_results["USG_L1"]= leader1_name, leader2_name
    df_results["LEADER 2"], df_results["USG_L2"] = leader1_usg, leader2_usg
    return df_results
